# tree_reentry_derisk_purged_bc_v1

Notebook-owned diagnostic for the active objective: improve strict OOS excess toward `+30pp`, keep MaxDD below `15%`, and avoid overfitting by using purged expanding-window validation before any promotion claim.

Governance:
- Canonical baseline: `runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv`, the current production-monitoring daily replay used by the previous re-entry diagnostics.
- Direct comparison baseline: `two_stage_reentry_derisk_purged_bc_v1`, because its notebook concluded that staged logistic BC is insufficient and recommended a non-linear tree/sequence learner before opening a PPO branch.
- Role: `diagnose`; no actor checkpoint and no runtime promotion from this notebook alone.
- Changed layer: `model` / `model_selection`; reusable purged split and scoring helpers stay in `src`, while candidate grids and tree settings stay in this notebook.
- Target windows: `wf_2022`, `wf_2023_2024`, shared bad-window clusters `C002_2025_rebound` and `C003_2026_reentry`.
- Gate: selected candidate must reach mean window delta `>= +30pp` and key-window MaxDD `<= 15%`; if it passes, it still needs saved-actor walk-forward validation before promotion.
- External references: Qlib (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`) for non-leaking workflow discipline; purged CV / embargo from Advances in Financial Machine Learning; behavior-cloning trading-agent reference (`https://arxiv.org/abs/2110.01325`); tree ensemble implementation basis from scikit-learn histogram gradient boosting and ExtraTrees docs.

Closest historical success case:
- `predictive_drawdown_risk_gate_v1` found strict OOS candidates with `relative_return > +30pp` and MaxDD near/under `15%`, but failed saved walk-forward gates.

Closest historical failure case:
- `two_stage_reentry_derisk_purged_bc_v1` improved mean delta to `+18.20pp` but still had key-window MaxDD `30.67%` and did not pass the combined gate.

Why this is not a duplicate:
- It keeps the same purged label discipline but replaces linear logistic BC with a compressed shallow tree ensemble screen and explicitly audits false positive de-risk/re-entry counts by window. The changed lever is model class and feature-set routing, not another threshold-only sweep. The grid is deliberately small so the notebook remains repeatable as a runtime artifact.


In [1]:
from __future__ import annotations

import json
import math
from dataclasses import asdict
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

from src.soxl_quant.app.budget_counterfactual import compute_metrics
from src.soxl_quant.app.oos_diagnostics import load_qlib_feature_frame
from src.soxl_quant.app.purged_walkforward import build_purged_walkforward_splits, single_position_frame
from src.soxl_quant.app.reentry_labels import ReentryOracleLabelConfig, build_reentry_oracle_label_frame

EXPERIMENT = "tree_reentry_derisk_purged_bc_v1"
OUTPUT_DIR = Path("runtime/model/card") / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_DAILY_CSV = Path("runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv")
LOGISTIC_SELECTION_JSON = Path("runtime/model/card/two_stage_reentry_derisk_purged_bc_v1/selection.json")
DAILY = pd.read_csv(BASELINE_DAILY_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
PRICE = DAILY.set_index("datetime")["price"].astype(float)
FEATURES = load_qlib_feature_frame(str(DAILY["datetime"].min().date()), str(DAILY["datetime"].max().date()))
FEATURES = FEATURES.reset_index().rename(columns={"index": "datetime"})
FEATURES["datetime"] = pd.to_datetime(FEATURES["datetime"])

WINDOWS = {
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "bad_C002_2025_rebound": ("2025-04-04", "2025-07-22"),
    "bad_C003_2026_reentry": ("2026-02-04", "2026-05-21"),
}
KEY_DD_WINDOWS = set(WINDOWS)

FEATURE_SETS = {
    "cluster_vol_reentry": [
        "vol_of_vol_21", "tracking_error_vol_21", "decay_63", "realized_vol_63", "tracking_error_21",
        "realized_vol_21", "mom_21", "mom_10", "relative_strength_21", "vix_percentile_252",
        "anchor_gap_180", "below_anchor_ratio_63", "regime_risk", "drawdown_63", "downside_semivol_21",
        "vix_zscore", "trend_gap_21", "trend_gap_63", "realized_vol_10", "days_since_high_63",
    ],
    "cluster_vol_compact": [
        "vol_of_vol_21", "tracking_error_vol_21", "decay_63", "realized_vol_63", "tracking_error_21",
        "realized_vol_21", "vix_percentile_252", "regime_risk", "drawdown_63", "days_since_high_63",
    ],
}
for name, cols in FEATURE_SETS.items():
    missing = [col for col in cols if col not in FEATURES.columns]
    if missing:
        raise ValueError(f"feature set {name} missing columns: {missing}")

logistic_selection = json.loads(LOGISTIC_SELECTION_JSON.read_text()) if LOGISTIC_SELECTION_JSON.exists() else {}
print({
    "daily_rows": len(DAILY),
    "daily_start": str(DAILY["datetime"].min().date()),
    "daily_end": str(DAILY["datetime"].max().date()),
    "feature_rows": len(FEATURES),
    "feature_sets": {name: len(cols) for name, cols in FEATURE_SETS.items()},
    "logistic_decision": logistic_selection.get("decision"),
    "logistic_best_mean_delta": logistic_selection.get("selected_candidate", {}).get("mean_delta_return"),
})

[8009:MainThread](2026-05-25 13:13:45,197) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[8009:MainThread](2026-05-25 13:13:45,869) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[8009:MainThread](2026-05-25 13:13:45,870) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[8009:MainThread](2026-05-25 13:13:48,710) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[8009:MainThread](2026-05-25 13:13:48,711) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[8009:MainThread](2026-05-25 13:13:48,713) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | Init data Done


{'daily_rows': 1605, 'daily_start': '2020-01-02', 'daily_end': '2026-05-21', 'feature_rows': 1605, 'feature_sets': {'cluster_vol_reentry': 20, 'cluster_vol_compact': 10}, 'logistic_decision': 'diagnostic_not_promoted', 'logistic_best_mean_delta': 18.197413391958435}


In [2]:
def score_positions(frame: pd.DataFrame, start: str, end: str, position_col: str = "position") -> dict:
    sub = frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].copy()
    return compute_metrics(single_position_frame(sub, position_col=position_col), PRICE.loc[start:end])


def base_window_table() -> pd.DataFrame:
    rows = []
    for name, (start, end) in WINDOWS.items():
        rows.append({"window": name, "variant": "baseline", **score_positions(DAILY, start, end)})
    return pd.DataFrame(rows)


BASE_WINDOWS = base_window_table()
BASE_WINDOWS[["window", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

,window,return,relative_return,max_dd,avg_pos,trades
0,wf_2022,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2023_2024,209.141657,17.368148,30.668184,0.497231,70
2,bad_C002_2025_rebound,76.882481,-124.606649,16.324924,0.572851,9
3,bad_C003_2026_reentry,17.799626,-217.582972,6.421473,0.161722,2


In [3]:
def build_labeled_feature_frame(
    cfg: ReentryOracleLabelConfig,
    *,
    derisk_dd: float,
    derisk_loss: float = -0.05,
    feature_cols: list[str],
) -> pd.DataFrame:
    labels = build_reentry_oracle_label_frame(DAILY, cfg)
    labels["derisk_label"] = (
        labels["has_forward_window"]
        & (labels["base_position"] >= 0.05)
        & ((labels["forward_drawdown"] >= float(derisk_dd)) | (labels["forward_return"] <= float(derisk_loss)))
    )
    labels["staged_reentry_label"] = labels["oracle_reentry"] & (~labels["derisk_label"])
    frame = labels.merge(FEATURES[["datetime", *feature_cols]], on="datetime", how="left")
    return frame.dropna(subset=feature_cols).reset_index(drop=True)


def _make_model(spec: dict, seed: int):
    kind = spec["kind"]
    if kind == "hgb":
        return HistGradientBoostingClassifier(
            max_iter=int(spec.get("max_iter", 80)),
            learning_rate=float(spec.get("learning_rate", 0.05)),
            max_leaf_nodes=int(spec.get("max_leaf_nodes", 7)),
            max_depth=spec.get("max_depth"),
            min_samples_leaf=int(spec.get("min_samples_leaf", 30)),
            l2_regularization=float(spec.get("l2_regularization", 0.1)),
            class_weight="balanced",
            random_state=seed,
        )
    if kind == "extra_trees":
        return ExtraTreesClassifier(
            n_estimators=int(spec.get("n_estimators", 120)),
            max_depth=spec.get("max_depth", 3),
            min_samples_leaf=int(spec.get("min_samples_leaf", 25)),
            max_features=float(spec.get("max_features", 0.7)),
            class_weight="balanced",
            random_state=seed,
            n_jobs=1,
        )
    raise ValueError(f"unknown model kind: {kind}")


def tree_probability(
    train: pd.DataFrame,
    test: pd.DataFrame,
    label_col: str,
    *,
    feature_cols: list[str],
    spec: dict,
    seed: int,
    min_positive_rows: int = 5,
) -> np.ndarray:
    y = train[label_col].astype(int)
    positives = int(y.sum())
    negatives = int((1 - y).sum())
    if positives < int(min_positive_rows) or negatives < int(min_positive_rows):
        return np.zeros(len(test), dtype=float)
    model = _make_model(spec, seed)
    model.fit(train[feature_cols], y)
    classes = list(model.classes_)
    if 1 not in classes:
        return np.zeros(len(test), dtype=float)
    return model.predict_proba(test[feature_cols])[:, classes.index(1)]


def apply_two_stage_path(
    test: pd.DataFrame,
    *,
    cfg: ReentryOracleLabelConfig,
    p_derisk: np.ndarray,
    p_reentry: np.ndarray,
    derisk_threshold: float,
    clear_threshold: float,
    reentry_threshold: float,
    derisk_target: float,
) -> pd.DataFrame:
    out = test[["datetime", "base_position", "derisk_label", "staged_reentry_label", "forward_return", "forward_drawdown"]].copy()
    out = out.rename(columns={"base_position": "position"})
    base = out["position"].to_numpy(dtype=float)
    derisk_mask = (p_derisk >= float(derisk_threshold)) & (base >= 0.05)
    reentry_mask = (
        (p_derisk <= float(clear_threshold))
        & (p_reentry >= float(reentry_threshold))
        & (base <= float(cfg.max_base_position))
        & (~derisk_mask)
    )
    out["base_position"] = base
    out.loc[derisk_mask, "position"] = np.minimum(out.loc[derisk_mask, "position"], float(derisk_target))
    out.loc[reentry_mask, "position"] = np.maximum(out.loc[reentry_mask, "position"], float(cfg.target_position))
    out["p_derisk"] = p_derisk
    out["p_reentry"] = p_reentry
    out["derisk_pred"] = derisk_mask
    out["reentry_pred"] = reentry_mask
    out["position_delta"] = out["position"] - out["base_position"]
    return out

In [4]:
def score_candidate(
    *,
    cfg: ReentryOracleLabelConfig,
    feature_set_name: str,
    feature_cols: list[str],
    model_name: str,
    model_spec: dict,
    derisk_dd: float,
    derisk_loss: float,
    derisk_threshold: float,
    clear_threshold: float,
    reentry_threshold: float,
    derisk_target: float,
) -> tuple[dict, list[dict], dict[str, pd.DataFrame], list[dict]]:
    frame = build_labeled_feature_frame(cfg, derisk_dd=derisk_dd, derisk_loss=derisk_loss, feature_cols=feature_cols)
    splits = build_purged_walkforward_splits(
        frame,
        WINDOWS,
        purge_rows=int(cfg.horizon_days),
        min_train_rows=120,
        min_test_rows=2,
    )
    row = {
        **{f"cfg_{k}": v for k, v in asdict(cfg).items()},
        "feature_set": feature_set_name,
        "feature_count": len(feature_cols),
        "model_name": model_name,
        "model_kind": model_spec["kind"],
        "derisk_dd": float(derisk_dd),
        "derisk_loss": float(derisk_loss),
        "derisk_threshold": float(derisk_threshold),
        "clear_threshold": float(clear_threshold),
        "reentry_threshold": float(reentry_threshold),
        "derisk_target": float(derisk_target),
        "split_count": len(splits),
        "total_derisk_predictions": 0,
        "total_reentry_predictions": 0,
        "total_derisk_true": 0,
        "total_reentry_true": 0,
    }
    comparison_rows = []
    audit_rows = []
    path_by_window: dict[str, pd.DataFrame] = {}
    deltas = []
    max_dds = []
    no_worse_windows = 0
    for split in splits:
        train = split.train[split.train["has_forward_window"]].copy()
        test = split.test.copy()
        p_derisk = tree_probability(train, test, "derisk_label", feature_cols=feature_cols, spec=model_spec, seed=11)
        p_reentry = tree_probability(train, test, "staged_reentry_label", feature_cols=feature_cols, spec=model_spec, seed=29)
        path = apply_two_stage_path(
            test,
            cfg=cfg,
            p_derisk=p_derisk,
            p_reentry=p_reentry,
            derisk_threshold=derisk_threshold,
            clear_threshold=clear_threshold,
            reentry_threshold=reentry_threshold,
            derisk_target=derisk_target,
        )
        path_by_window[split.name] = path
        start, end = WINDOWS[split.name]
        metrics = score_positions(path, start, end)
        base = BASE_WINDOWS[BASE_WINDOWS["window"] == split.name].iloc[0].to_dict()
        delta_return = metrics["return"] - float(base["return"])
        delta_relative = metrics["relative_return"] - float(base["relative_return"])
        deltas.append(delta_relative)
        max_dds.append(metrics["max_dd"])
        if delta_relative >= -1e-9 and metrics["max_dd"] <= float(base["max_dd"]) + 1e-9:
            no_worse_windows += 1
        comparison_rows.append({"window": split.name, "variant": "baseline", **base})
        comparison_rows.append({"window": split.name, "variant": "tree_two_stage", **metrics, "delta_return": delta_return, "delta_relative_return": delta_relative})
        derisk_pred = path["derisk_pred"].astype(bool)
        reentry_pred = path["reentry_pred"].astype(bool)
        derisk_true = path["derisk_label"].astype(bool)
        reentry_true = path["staged_reentry_label"].astype(bool)
        audit_rows.append({
            "window": split.name,
            "train_rows": int(len(train)),
            "test_rows": int(len(test)),
            "derisk_pred": int(derisk_pred.sum()),
            "derisk_true": int(derisk_true.sum()),
            "derisk_tp": int((derisk_pred & derisk_true).sum()),
            "derisk_fp": int((derisk_pred & ~derisk_true).sum()),
            "reentry_pred": int(reentry_pred.sum()),
            "reentry_true": int(reentry_true.sum()),
            "reentry_tp": int((reentry_pred & reentry_true).sum()),
            "reentry_fp": int((reentry_pred & ~reentry_true).sum()),
        })
        row["total_derisk_predictions"] += int(derisk_pred.sum())
        row["total_reentry_predictions"] += int(reentry_pred.sum())
        row["total_derisk_true"] += int(derisk_true.sum())
        row["total_reentry_true"] += int(reentry_true.sum())
    row["mean_delta_relative_return"] = float(np.mean(deltas)) if deltas else float("nan")
    row["min_delta_relative_return"] = float(np.min(deltas)) if deltas else float("nan")
    row["max_key_window_dd"] = float(np.max(max_dds)) if max_dds else float("nan")
    row["return_gate"] = bool(deltas and row["mean_delta_relative_return"] >= 30.0)
    row["maxdd_gate"] = bool(max_dds and row["max_key_window_dd"] <= 15.0)
    row["no_worse_window_count"] = int(no_worse_windows)
    row["no_worse_gate"] = bool(no_worse_windows == len(splits) and len(splits) == len(WINDOWS))
    row["full_gate"] = bool(row["return_gate"] and row["maxdd_gate"] and row["no_worse_gate"])
    # Prefer real uplift and low drawdown, but penalize sparse/no-action candidates and false positives.
    fp_penalty = sum(a["derisk_fp"] + a["reentry_fp"] for a in audit_rows) * 0.05
    action_penalty = 15.0 if row["total_derisk_predictions"] + row["total_reentry_predictions"] == 0 else 0.0
    row["diagnostic_score"] = float(row["mean_delta_relative_return"] - 1.5 * max(0.0, row["max_key_window_dd"] - 15.0) - fp_penalty - action_penalty)
    return row, comparison_rows, path_by_window, audit_rows

In [5]:
GRID_CONFIGS = [
    ReentryOracleLabelConfig(21, 0.65, 0.75, 0.40, 0.18),
    ReentryOracleLabelConfig(21, 0.55, 0.75, 0.40, 0.18),
]
MODEL_SPECS = {
    "hgb_tiny_l2": {"kind": "hgb", "max_iter": 40, "learning_rate": 0.05, "max_leaf_nodes": 7, "min_samples_leaf": 40, "l2_regularization": 1.0},
    "extra_trees_tiny": {"kind": "extra_trees", "n_estimators": 60, "max_depth": 3, "min_samples_leaf": 35, "max_features": 0.7},
}
THRESHOLD_PAIRS = [
    (0.65, 0.65),
    (0.75, 0.75),
]

rows = []
comparison_by_key = {}
path_by_key = {}
audit_by_key = {}
for cfg, (feature_set_name, feature_cols), (model_name, model_spec), derisk_dd, (derisk_threshold, reentry_threshold) in product(
    GRID_CONFIGS,
    FEATURE_SETS.items(),
    MODEL_SPECS.items(),
    [0.15],
    THRESHOLD_PAIRS,
):
    clear_threshold = min(float(derisk_threshold), 0.55)
    row, comparison_rows, paths, audit_rows = score_candidate(
        cfg=cfg,
        feature_set_name=feature_set_name,
        feature_cols=feature_cols,
        model_name=model_name,
        model_spec=model_spec,
        derisk_dd=derisk_dd,
        derisk_loss=-0.05,
        derisk_threshold=derisk_threshold,
        clear_threshold=clear_threshold,
        reentry_threshold=reentry_threshold,
        derisk_target=0.0,
    )
    key = len(rows)
    row["candidate_key"] = key
    rows.append(row)
    comparison_by_key[key] = comparison_rows
    path_by_key[key] = paths
    audit_by_key[key] = audit_rows

results = pd.DataFrame(rows).sort_values(["full_gate", "diagnostic_score", "mean_delta_relative_return"], ascending=[False, False, False]).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "tree_grid_results.csv", index=False)
print("grid rows", len(results), "full gate passes", int(results["full_gate"].sum()))
results.head(15)[[
    "candidate_key", "model_name", "feature_set", "cfg_horizon_days", "cfg_max_base_position", "cfg_target_position",
    "derisk_dd", "derisk_threshold", "clear_threshold", "reentry_threshold", "mean_delta_relative_return",
    "min_delta_relative_return", "max_key_window_dd", "return_gate", "maxdd_gate", "no_worse_gate", "full_gate",
    "total_derisk_predictions", "total_reentry_predictions", "diagnostic_score",
]]

grid rows 16 full gate passes 0


,candidate_key,model_name,feature_set,cfg_horizon_days,cfg_max_base_position,cfg_target_position,derisk_dd,derisk_threshold,clear_threshold,reentry_threshold,mean_delta_relative_return,min_delta_relative_return,max_key_window_dd,return_gate,maxdd_gate,no_worse_gate,full_gate,total_derisk_predictions,total_reentry_predictions,diagnostic_score
0,3,extra_trees_tiny,cluster_vol_reentry,21,0.65,0.75,0.15,0.75,0.55,0.75,1.320653,0.000000,31.096849,False,False,False,False,0,10,-22.924620
1,11,extra_trees_tiny,cluster_vol_reentry,21,0.55,0.75,0.15,0.75,0.55,0.75,1.320653,0.000000,31.096849,False,False,False,False,0,10,-22.924620
2,13,hgb_tiny_l2,cluster_vol_compact,21,0.55,0.75,0.15,0.75,0.55,0.75,6.254463,-6.148536,34.252015,False,False,False,False,72,29,-25.273560
3,14,extra_trees_tiny,cluster_vol_compact,21,0.55,0.75,0.15,0.65,0.55,0.65,-0.788416,-6.424143,31.096849,False,False,False,False,13,18,-25.583690
4,6,extra_trees_tiny,cluster_vol_compact,21,0.65,0.75,0.15,0.65,0.55,0.65,-0.995488,-6.424143,31.096849,False,False,False,False,13,16,-25.790762
5,8,hgb_tiny_l2,cluster_vol_reentry,21,0.55,0.75,0.15,0.65,0.55,0.65,5.398242,-4.445965,34.401483,False,False,False,False,117,16,-26.353983
6,5,hgb_tiny_l2,cluster_vol_compact,21,0.65,0.75,0.15,0.75,0.55,0.75,6.828584,-5.279094,36.495015,False,False,False,False,72,27,-28.063938
7,0,hgb_tiny_l2,cluster_vol_reentry,21,0.65,0.75,0.15,0.65,0.55,0.65,5.061496,-5.108666,35.584807,False,False,False,False,117,20,-28.615714
8,9,hgb_tiny_l2,cluster_vol_reentry,21,0.55,0.75,0.15,0.75,0.55,0.75,-3.423638,-32.651561,34.778513,False,False,False,False,15,15,-33.941407
9,1,hgb_tiny_l2,cluster_vol_reentry,21,0.65,0.75,0.15,0.75,0.55,0.75,-3.949478,-32.651561,36.626314,False,False,False,False,15,18,-37.388949


In [6]:
selected = results.iloc[0].to_dict()
selected_key = int(selected["candidate_key"])
selected_comparison = pd.DataFrame(comparison_by_key[selected_key])
selected_audit = pd.DataFrame(audit_by_key[selected_key])
selected_comparison.to_csv(OUTPUT_DIR / "selected_comparison.csv", index=False)
selected_audit.to_csv(OUTPUT_DIR / "selected_signal_audit.csv", index=False)

selected_paths = []
for window, frame in path_by_key[selected_key].items():
    out = frame.copy()
    out["window"] = window
    selected_paths.append(out)
if selected_paths:
    pd.concat(selected_paths, ignore_index=True).to_csv(OUTPUT_DIR / "selected_paths.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": "candidate_requires_saved_walkforward" if bool(selected["full_gate"]) else "diagnostic_not_promoted",
    "reason": (
        "tree two-stage candidate clears notebook gates but still needs saved-actor walk-forward validation"
        if bool(selected["full_gate"])
        else "tree two-stage candidates still fail the combined +30pp mean relative return, <=15% key-window MaxDD, and no-worse-window gates"
    ),
    "baseline_daily_csv": str(BASELINE_DAILY_CSV),
    "direct_comparison": str(LOGISTIC_SELECTION_JSON),
    "grid_results_csv": str(OUTPUT_DIR / "tree_grid_results.csv"),
    "selected_comparison_csv": str(OUTPUT_DIR / "selected_comparison.csv"),
    "selected_signal_audit_csv": str(OUTPUT_DIR / "selected_signal_audit.csv"),
    "selected_paths_csv": str(OUTPUT_DIR / "selected_paths.csv"),
    "selected_candidate": {k: (None if pd.isna(v) else v) for k, v in selected.items() if isinstance(v, (int, float, str, bool)) or pd.isna(v)},
    "logistic_reference": logistic_selection.get("selected_candidate", {}),
    "next_path": (
        "run saved-actor walk-forward packaging before any promotion claim"
        if bool(selected["full_gate"])
        else "tree model class alone does not solve the overfit/risk gap; inspect label precision and move toward explicit drawdown-state features or sequence model only if false positives are reducible"
    ),
}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")

selected_comparison[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]].sort_values(["window", "variant"])

,window,variant,return,relative_return,max_dd,avg_pos,trades
4,bad_C002_2025_rebound,baseline,76.882481,-124.606649,16.324924,0.572851,9
5,bad_C002_2025_rebound,tree_two_stage,82.165095,-119.324035,18.931472,0.607175,12
6,bad_C003_2026_reentry,baseline,17.799626,-217.582972,6.421473,0.161722,2
7,bad_C003_2026_reentry,tree_two_stage,17.799626,-217.582972,6.421473,0.161722,2
0,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2022,tree_two_stage,-21.567182,65.020890,31.096849,0.248277,26
2,wf_2023_2024,baseline,209.141657,17.368148,30.668184,0.497231,70
3,wf_2023_2024,tree_two_stage,209.141657,17.368148,30.668184,0.497231,70


In [7]:
selected_audit

,window,train_rows,test_rows,derisk_pred,derisk_true,derisk_tp,derisk_fp,reentry_pred,reentry_true,reentry_tp,reentry_fp
0,wf_2022,484,251,0,188,0,0,0,26,0,0
1,wf_2023_2024,735,502,0,231,0,0,0,26,0,0
2,bad_C002_2025_rebound,1300,74,0,11,0,0,10,25,8,2
3,bad_C003_2026_reentry,1509,75,0,16,0,0,0,19,0,0


In [8]:
# Local feature-importance audit for the selected model family on the most recent train/test split.
# This is diagnostic only; it is not treated as a promoted factor list.
importance_rows = []
selected_feature_cols = FEATURE_SETS[str(selected["feature_set"])]
selected_cfg = ReentryOracleLabelConfig(
    int(selected["cfg_horizon_days"]),
    float(selected["cfg_max_base_position"]),
    float(selected["cfg_target_position"]),
    float(selected["cfg_min_forward_return"]),
    float(selected["cfg_max_forward_drawdown"]),
)
selected_spec = MODEL_SPECS[str(selected["model_name"])]
feature_frame = build_labeled_feature_frame(selected_cfg, derisk_dd=float(selected["derisk_dd"]), derisk_loss=float(selected["derisk_loss"]), feature_cols=selected_feature_cols)
splits = build_purged_walkforward_splits(feature_frame, WINDOWS, purge_rows=int(selected_cfg.horizon_days), min_train_rows=120, min_test_rows=2)
for label_col, seed in [("derisk_label", 11), ("staged_reentry_label", 29)]:
    # Use the latest split with enough class balance to avoid fitting a degenerate diagnostic model.
    for split in reversed(splits):
        train = split.train[split.train["has_forward_window"]].copy()
        test = split.test.copy()
        y_train = train[label_col].astype(int)
        y_test = test[label_col].astype(int)
        if int(y_train.sum()) >= 5 and int((1-y_train).sum()) >= 5 and y_test.nunique() > 1:
            model = _make_model(selected_spec, seed)
            model.fit(train[selected_feature_cols], y_train)
            perm = permutation_importance(model, test[selected_feature_cols], y_test, n_repeats=4, random_state=seed, scoring="roc_auc")
            for feature, mean, std in zip(selected_feature_cols, perm.importances_mean, perm.importances_std):
                importance_rows.append({
                    "label": label_col,
                    "split": split.name,
                    "feature": feature,
                    "importance_mean": float(mean),
                    "importance_std": float(std),
                })
            break
importance = pd.DataFrame(importance_rows).sort_values(["label", "importance_mean"], ascending=[True, False]) if importance_rows else pd.DataFrame()
importance.to_csv(OUTPUT_DIR / "selected_permutation_importance.csv", index=False)
importance.head(20)

,label,split,feature,importance_mean,importance_std
11,derisk_label,bad_C003_2026_reentry,below_anchor_ratio_63,0.057733,0.005053
19,derisk_label,bad_C003_2026_reentry,days_since_high_63,0.025424,0.011110
16,derisk_label,bad_C003_2026_reentry,trend_gap_21,0.024894,0.002309
10,derisk_label,bad_C003_2026_reentry,anchor_gap_180,0.013506,0.016537
18,derisk_label,bad_C003_2026_reentry,realized_vol_10,0.002913,0.014338
1,derisk_label,bad_C003_2026_reentry,tracking_error_vol_21,0.001059,0.001835
9,derisk_label,bad_C003_2026_reentry,vix_percentile_252,0.000530,0.009781
8,derisk_label,bad_C003_2026_reentry,relative_strength_21,0.000000,0.000000
2,derisk_label,bad_C003_2026_reentry,decay_63,-0.000530,0.000917
4,derisk_label,bad_C003_2026_reentry,tracking_error_21,-0.001589,0.000917


In [9]:
best = results.iloc[0]
pass_count = int(results["full_gate"].sum())
logistic_best = logistic_selection.get("selected_candidate", {})
logistic_mean = logistic_best.get("mean_delta_return")
logistic_dd = logistic_best.get("max_key_window_dd")
summary = {
    "candidate_count": int(len(results)),
    "full_gate_pass_count": pass_count,
    "best_mean_delta_relative_return": float(best["mean_delta_relative_return"]),
    "best_min_delta_relative_return": float(best["min_delta_relative_return"]),
    "best_max_key_window_dd": float(best["max_key_window_dd"]),
    "best_total_derisk_predictions": int(best["total_derisk_predictions"]),
    "best_total_reentry_predictions": int(best["total_reentry_predictions"]),
    "logistic_reference_mean_delta": None if logistic_mean is None else float(logistic_mean),
    "logistic_reference_max_key_window_dd": None if logistic_dd is None else float(logistic_dd),
}
conclusion = f"""## Runtime Conclusion

Decision: `{selection['decision']}`.

Screened `{summary['candidate_count']}` shallow tree/ensemble two-stage candidates with purged expanding-window evaluation; full gate pass count `{summary['full_gate_pass_count']}`. Best candidate mean relative-return delta `{summary['best_mean_delta_relative_return']:.2f}pp`, min window delta `{summary['best_min_delta_relative_return']:.2f}pp`, and max key-window DD `{summary['best_max_key_window_dd']:.2f}%`.

Direct comparison to previous logistic BC: logistic best mean delta `{summary['logistic_reference_mean_delta']:.2f}pp` and max key-window DD `{summary['logistic_reference_max_key_window_dd']:.2f}%` if available. The tree screen improves only if it raises mean delta without increasing key-window drawdown; otherwise the model-class hypothesis should be dropped.

Mechanism readout: non-linear trees can use volatility/tracking-error interactions, but the selected path must still pass the same `+30pp` / `15%` notebook gate and the no-worse-window guard. False positives in `selected_signal_audit.csv` are the main overfit check; a candidate with many actions and weak label precision is not a safe warm-start target.

Keep/drop: keep this as diagnostic evidence. Do not promote a tree BC policy directly from notebook replay. If the candidate is not gate-clean, the next useful loop is not another shallow model sweep; it should add explicit drawdown-state/sequence features or improve label quality before retraining.

Artifacts: `tree_grid_results.csv`, `selected_comparison.csv`, `selected_signal_audit.csv`, `selected_paths.csv`, `selected_permutation_importance.csv`, and `selection.json`.
"""
display(Markdown(conclusion))
print(json.dumps(summary, indent=2))

## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

Screened `16` shallow tree/ensemble two-stage candidates with purged expanding-window evaluation; full gate pass count `0`. Best candidate mean relative-return delta `1.32pp`, min window delta `0.00pp`, and max key-window DD `31.10%`.

Direct comparison to previous logistic BC: logistic best mean delta `18.20pp` and max key-window DD `30.67%` if available. The tree screen improves only if it raises mean delta without increasing key-window drawdown; otherwise the model-class hypothesis should be dropped.

Mechanism readout: non-linear trees can use volatility/tracking-error interactions, but the selected path must still pass the same `+30pp` / `15%` notebook gate and the no-worse-window guard. False positives in `selected_signal_audit.csv` are the main overfit check; a candidate with many actions and weak label precision is not a safe warm-start target.

Keep/drop: keep this as diagnostic evidence. Do not promote a tree BC policy directly from notebook replay. If the candidate is not gate-clean, the next useful loop is not another shallow model sweep; it should add explicit drawdown-state/sequence features or improve label quality before retraining.

Artifacts: `tree_grid_results.csv`, `selected_comparison.csv`, `selected_signal_audit.csv`, `selected_paths.csv`, `selected_permutation_importance.csv`, and `selection.json`.


{
  "candidate_count": 16,
  "full_gate_pass_count": 0,
  "best_mean_delta_relative_return": 1.3206534565801142,
  "best_min_delta_relative_return": 0.0,
  "best_max_key_window_dd": 31.096849210284773,
  "best_total_derisk_predictions": 0,
  "best_total_reentry_predictions": 10,
  "logistic_reference_mean_delta": 18.197413391958435,
  "logistic_reference_max_key_window_dd": 30.66818414624621
}
